# FinMate — QLoRA supervised fine-tuning (template)

1. Upload `training/data/example_sft.jsonl` (or your full dataset) to Colab / Drive.
2. Set `BASE_MODEL` and training hyperparameters below.
3. Save the LoRA adapter to Drive and document the commit hash + hyperparameters in your report.

**Note:** Training runs on Colab; load the adapter in `backend` later via Hugging Face `PeftModel.from_pretrained` + 4-bit base.

In [ ]:
# Optional: mount Drive for checkpoints
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl sentencepiece protobuf

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"  # or another instruct model you have access to
DATA_PATH = "/content/example_sft.jsonl"  # change to your path

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")


def formatting_prompts(examples):
    texts = []
    for msgs in examples["messages"]:
        parts = []
        for m in msgs:
            role = m["role"]
            content = m["content"]
            if role == "system":
                parts.append(f"<s>[SYSTEM]\n{content}\n")
            elif role == "user":
                parts.append(f"[USER]\n{content}\n")
            elif role == "assistant":
                parts.append(f"[ASSISTANT]\n{content}</s>")
        texts.append("".join(parts))
    return {"text": texts}


dataset = dataset.map(formatting_prompts, batched=True, remove_columns=dataset.column_names)

In [ ]:
args = SFTConfig(
    output_dir="./finmate-lora-out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    max_seq_length=2048,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=args,
)
trainer.train()
trainer.model.save_pretrained("./finmate-lora-out/final_adapter")
tokenizer.save_pretrained("./finmate-lora-out/final_adapter")